## ⚙️ Setup Instructions

This notebook expects the dataset and precomputed embeddings to be available 
in your Google Drive under a specific path.

**Before running:**
1. Download the dataset and embeddings from the links below and upload them 
   to your own Google Drive:
   - 📊 Dataset: [https://drive.google.com/file/d/1G5IoNINVZN-YgWOPzT_wzZis5R7dKBQk/view?usp=drive_link]
   - 🧠 Embeddings: [https://drive.google.com/file/d/1wbszyyArl-4uanOcZsyt_MMM3D4QYC8h/view?usp=sharing]
2. Place them in your Drive so the path matches the one used in the code 
   below (e.g. `MyDrive/MyDrive/`), **or** edit the file paths in 
   the next cell to match wherever you saved them.
3. Run the cells in order — the first cell will prompt you to authorize 
   Google Drive access.

In [1]:
 !pip install -q sentence-transformers faiss-cpu pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 26.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import faiss
import json
import re
from sentence_transformers import SentenceTransformer

df  = pd.read_csv("/content/drive/MyDrive/MyDrive/unified_dataset.csv")
embeddings = np.load("/content/drive/MyDrive/MyDrive/embeddings.npy").astype("float32")

/tmp/ipykernel_1298/1119365283.py:8: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  df  = pd.read_csv("/content/drive/MyDrive/MyDrive/unified_dataset.csv")


In [4]:
def clean_genres(genres_str):
    if pd.isna(genres_str):
        return "Unknown"
    parts = [g.strip() for g in str(genres_str).split(",")]
    seen, result = set(), []
    for p in parts:
        if p and p not in seen:
            seen.add(p)
            result.append(p)
        if len(result) == 4:
            break
    return ", ".join(result) if result else "Unknown"

df["genres"] = df["genres"].apply(clean_genres)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print(f" Dataset loaded: {len(df):,} entries")
print(df["type"].value_counts().to_string())



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Dataset loaded: 572,049 entries
type
movie      471373
tv_show     85755
book        14921


In [5]:
MOOD_MAP = {
    "زهقان": {
        "keywords": ["زهقان", "زهقانة", "طفشان", "ممل", "بدي اشي خفيف"],
        "genres":   ["Comedy", "Animation", "Adventure"],
        "boost":    "خفيف وكوميدي",
        "english":  "entertaining light-hearted fun comedy",
    },
    "حزين": {
        "keywords": ["حزين", "حزينة", "مكتئب", "بدي ابكي", "دراما"],
        "genres":   ["Drama", "Romance"],
        "boost":    "مؤثر وعاطفي",
        "english":  "emotional touching dramatic heartfelt",
    },
    "متحمس": {
        "keywords": ["متحمس", "قوي", "أكشن", "قتال", "حماس", "مغامرة", "شرطة", "مطاردات", "سريع"],
        "genres":   ["Action", "Adventure", "Sci-Fi", "Thriller", "Crime"],
        "boost":    "أكشن قوي ومثير",
        "english":  "action-packed thrilling exciting adventure",
    },
    "خايف": {
        "keywords": ["خايف", "خايفة", "رعب", "مرعب", "بخوف"],
        "genres":   ["Horror", "Thriller"],
        "boost":    "مرعب ومتوتر",
        "english":  "scary horror frightening terrifying",
    },
    "بدي أضحك": {
        "keywords": ["بدي اضحك", "مضحك", "كوميدي", "نكت"],
        "genres":   ["Comedy", "Animation"],
        "boost":    "كوميدي وممتع",
        "english":  "comedy funny humorous laugh",
    },
    "متوتر": {
        "keywords": ["متوتر", "تشويق", "غموض", "جريمة", "تحقيق", "عميق"],
        "genres":   ["Thriller", "Mystery", "Crime"],
        "boost":    "غموض وتشويق",
        "english":  "suspenseful mysterious tense crime investigation",
    },
    "رومانسي": {
        "keywords": ["حب", "رومانسي", "عاطفي", "علاقة"],
        "genres":   ["Romance", "Drama"],
        "boost":    "رومانسي وعاطفي",
        "english":  "romantic love relationship passion",
    },
}

CONTENT_MAP = {
    "فيلم":  "movie",
    "مسلسل": "tv_show",
    "رواية": "book",
    "كتاب":  "book",
}


In [6]:

def detect_mood(user_input):
    text = user_input.lower()
    for mood, data in MOOD_MAP.items():
        for word in data["keywords"]:
            if word in text:
                return mood
    return None



In [7]:
def preprocess(user_input, mood_map):
    text = user_input.lower().strip()
    NEGATION_WORDS = ["لا", "مش", "مو", "not", "ما"]

    words         = text.split()
    cleaned_words = []
    negated_words = []
    skip_next     = False

    for w in words:
        if w in NEGATION_WORDS:
            skip_next = True
            continue
        if skip_next:
            negated_words.append(w)
            skip_next = False
            continue
        cleaned_words.append(w)

    cleaned_text   = " ".join(cleaned_words)
    detected_moods = []

    for mood, data in mood_map.items():
        for keyword in data["keywords"]:
            if keyword in cleaned_text:
                detected_moods.append(mood)
                break

    return cleaned_text, detected_moods, negated_words

In [8]:
def apply_constraints(results, constraints):
    if constraints["year_from"]:
        results = results[results["year"] >= constraints["year_from"]]
    if constraints["year_to"]:
        results = results[results["year"] <= constraints["year_to"]]
    if constraints["min_rating"]:
        results = results[results["rating"] >= constraints["min_rating"]]
    if constraints["max_runtime"]:
        results = results[results["runtime"] <= constraints["max_runtime"]]
    return results


In [9]:
KEYWORD_MAP = {
    "شرطة":   "police detective law enforcement",
    "تحقيق":  "investigation crime solving detective",
    "جريمة":  "crime murder thriller criminal",
    "غموض":   "mystery suspense unknown secret",
    "عميق":   "deep psychological complex thought-provoking",
    "مغامرة": "adventure journey quest exploration",
    "عائلة":  "family relationships home",
    "حب":     "love romance relationship",
    "روبوت":  "robot artificial intelligence technology",
    "فضاء":   "space science fiction galaxy",
    "تاريخ":  "historical period drama ancient",
    "حرب":    "war conflict battle military",
    "أطفال":  "children family animation kids",
    "خيال":   "fantasy magical supernatural",
    "مستقبل": "future dystopia science fiction",
    "طلاب":   "students school college youth",
    "طالبات": "female students school girls",
    "اصدقاء": "friends friendship group",
}

def rewrite_query(topic, mood, content_type):
    """
    يحوّل الـ belief state لـ query إنجليزي دقيق للـ semantic search
    """
    parts = []

    # 1) الموضوع → استخرج الكلمات المفتاحية
    if topic:
        topic_english = []
        for arabic, english in KEYWORD_MAP.items():
            if arabic in topic:
                topic_english.append(english)
        if topic_english:
            parts.extend(topic_english)
        else:
            # لو ما في keyword معروف → استخدم الـ topic مباشرة
            parts.append(topic)

    #  المود → كلمات إنجليزية دقيقة
    if mood and mood in MOOD_MAP:
        parts.append(MOOD_MAP[mood]["english"])

    #  النوع → context
    type_context = {
        "movie":   "film movie",
        "tv_show": "TV series show episodes",
        "book":    "novel book fiction story",
    }
    if content_type and content_type in type_context:
        parts.append(type_context[content_type])

    if not parts:
        return ""

    return " ".join(parts)


In [10]:
class DialogMemory:
    def __init__(self):
        self.state = {
            "mood":         None,
            "content_type": None,
            "topic":        None,
            "constraints": {
                "year_from":  None,
                "year_to":    None,
                "min_rating": None,
                "max_runtime": None,
                "language":   None,
            }
        }
        self.history     = []
        self.last_results = None

    def extract_constraints(self, text):
        # استخراج السنة
        years = re.findall(r"\d{4}", text)
        if years:
            year = int(years[0])
            if any(w in text for w in ["بعد", "ابتداء", "من سنة"]):
                self.state["constraints"]["year_from"] = year + 1
            elif "قبل" in text:
                self.state["constraints"]["year_to"] = year - 1

        # استخراج التقييم
        if any(w in text for w in ["تقييم", "rating", "تقييمو", "فوق", "أكثر من"]):
            nums = re.findall(r"\d+\.?\d*", text)
            if nums:
                self.state["constraints"]["min_rating"] = float(nums[0])

        # استخراج المدة
        if "ساعتين" in text:
            self.state["constraints"]["max_runtime"] = 120
        elif "ساعة" in text:
            self.state["constraints"]["max_runtime"] = 60

    def update(self, user_text):
        text = user_text.strip()

        # تحديث الـ Mood
        detected_mood = detect_mood(text)
        if detected_mood:
            self.state["mood"] = detected_mood

        # تحديث نوع المحتوى
        for word, ctype in CONTENT_MAP.items():
            if word in text:
                self.state["content_type"] = ctype
                break

        # تحديث الـ topic بذكاء
        story_triggers = ["عن", "قصة", "يحكي", "بتحكي", "فيها", "about", "بدي اشي"]
        content_words  = list(CONTENT_MAP.keys())
        negation_words = ["ما بدي", "مش", "لا", "مو"]

        has_story   = any(t in text for t in story_triggers)
        has_negation = any(n in text for n in negation_words)
        # نفي على النوع فقط؟ "ما بدي فيلم بدي مسلسل"
        negation_on_type_only = (
            has_negation and
            any(c in text for c in content_words) and
            not has_story
        )

        if has_story:
            # دايمًا احفظ الـ topic لو في قصة، حتى لو في نفي على شي تاني
            self.state["topic"] = text
        elif negation_on_type_only:
            # غيّر النوع بس، ما تمس الـ topic
            pass
        # غير هيك → ما تغير الـ topic

        # استخراج القيود
        self.extract_constraints(text)

        # احفظ في الـ History
        self.history.append({"role": "user", "text": text})

    def add_system_message(self, text):
        self.history.append({"role": "system", "text": text})

    def needs_clarification(self):
        if not self.state["topic"] and not self.state["mood"]:
            return "كيف مودك هلقيت؟ 😊\n(مثال: زهقان / حزين / متحمس / بدي أضحك)"
        if self.state["mood"] and not self.state["content_type"]:
            return "حلو! تحب فيلم، مسلسل، ولا رواية؟ 🎬📺📚"
        if self.state["topic"] and not self.state["content_type"]:
            return "شو تفضل؟ فيلم، مسلسل، ولا رواية؟ 🎬📺📚"
        return None

    def build_query(self):
        query = rewrite_query(
            self.state.get("topic"),
            self.state.get("mood"),
            self.state.get("content_type"),
        )
        # fallback: آخر رسالة من المستخدم
        if not query:
            user_msgs = [m["text"] for m in self.history if m["role"] == "user"]
            if user_msgs:
                query = user_msgs[-1]
        return query



In [11]:
def search_enhanced(query, memory, top_k=200):
    q_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(q_vec, min(top_k, index.ntotal))

    results = df.iloc[indices[0]].copy()
    results["distance"]  = distances[0]
    results["relevance"] = 1 / (1 + results["distance"])

    # فلتر الـ garbage  overview غير منطقية
    results = results[
        results["overview"].apply(lambda x:
            isinstance(x, str) and
            len(x) > 50 and
            len(set(x.split())) > 8
        )
    ]
    results = results[results["rating"] > 0]

    results = apply_constraints(results, memory.state["constraints"])
    return results


def apply_filters(results, state, weight=2.5):
    if results.empty:
        return results

    filtered = results.copy()

    #  فلتر النوع
    if state.get("content_type"):
        type_filtered = filtered[filtered["type"] == state["content_type"]].copy()
        if not type_filtered.empty:
            filtered = type_filtered

    #  Rating score
    if "rating" in filtered.columns:
        filtered["rating_score"] = filtered["rating"] / 10

    #  Genre boost حسب الـ mood
    if state.get("mood") in MOOD_MAP:
        genres = MOOD_MAP[state["mood"]]["genres"]

        def genre_score(genres_str):
            if pd.isna(genres_str):
                return 0
            g = str(genres_str).lower()
            matches = sum(1 for genre in genres if genre.lower() in g)
            return matches / len(genres)

        filtered.loc[:, "genre_match"]   = filtered["genres"].apply(genre_score)
        filtered.loc[:, "boosted_score"] = (
            filtered["relevance"] * (1 + weight * filtered["genre_match"])
        )

    #  ترتيب
    if "boosted_score" in filtered.columns:
        filtered["final_score"] = (
            0.5 * filtered["relevance"] +
            0.3 * filtered["boosted_score"] +
            0.2 * filtered.get("rating_score", 0)
        )
        filtered = filtered.sort_values("final_score", ascending=False)
    else:
        filtered = filtered.sort_values("relevance", ascending=False)

    return filtered


def rerank(results, state, top_k=5):
    """
    Reranker: يرتب الـ Top 20 من apply_filters بـ final score مركّب
    """
    if results.empty:
        return results

    scored = results.head(20).copy()

    # Semantic similarity — 40%
    scored["sim_score"] = scored["relevance"]

    # Genre match — 30%
    genres_wanted = []
    if state.get("mood") and state["mood"] in MOOD_MAP:
        genres_wanted = MOOD_MAP[state["mood"]]["genres"]

    def genre_match_score(g):
        if not genres_wanted or pd.isna(g):
            return 0
        g_lower = str(g).lower()
        matches = sum(1 for genre in genres_wanted if genre.lower() in g_lower)
        return matches / len(genres_wanted)

    scored["genre_score"] = scored["genres"].apply(genre_match_score)

    # Rating — 20%
    max_rating = scored["rating"].max() if scored["rating"].max() > 0 else 10
    scored["rating_score"] = scored["rating"] / max_rating

    # Popularity — 10%
    if "vote_count" in scored.columns and scored["vote_count"].max() > 0:
        scored["pop_score"] = scored["vote_count"] / scored["vote_count"].max()
    else:
        scored["pop_score"] = 0

    # Final score
    scored["final_score"] = (
        0.40 * scored["sim_score"] +
        0.30 * scored["genre_score"] +
        0.20 * scored["rating_score"] +
        0.10 * scored["pop_score"]
    )

    return scored.sort_values("final_score", ascending=False).head(top_k)


In [12]:
def display_results_formatted(results, top_k=5, full=False):
    if results.empty:
        print("\nNo suitable results found. Please try different criteria.")
        return

    count = len(results) if full else min(top_k, len(results))
    shown = results.head(count)

    print("\n" + "="*70)
    print(f"Search Results (Top {count} of {len(results)})")
    print("="*70 + "\n")

    type_map = {"movie": "Film", "tv_show": "Series", "book": "Book"}

    for idx, (_, row) in enumerate(shown.iterrows(), 1):
        title        = row.get("title", "Unknown")
        year         = int(row["year"]) if pd.notna(row.get("year")) else "N/A"
        rating       = row.get("rating", 0)
        rating_str   = f"{rating:.1f}" if pd.notna(rating) else "N/A"
        content_type = row.get("type", "")
        type_name    = type_map.get(content_type, content_type)

        # Genres — أول 4 بس
        genres_raw  = str(row.get("genres", ""))
        genres_list = [g.strip() for g in genres_raw.split(",") if g.strip()]
        seen_g, result_g = set(), []
        for g in genres_list:
            if g not in seen_g:
                seen_g.add(g)
                result_g.append(g)
            if len(result_g) == 4:
                break
        genres = ", ".join(result_g) if result_g else "N/A"

        # Details حسب النوع
        extra = []
        if content_type == "tv_show" and pd.notna(row.get("num_seasons")):
            extra.append(f"{int(row['num_seasons'])} seasons")
        elif content_type == "movie" and pd.notna(row.get("runtime")):
            extra.append(f"{int(row['runtime'])} min")
        elif content_type == "book" and pd.notna(row.get("num_pages")):
            extra.append(f"{int(row['num_pages'])} pages")

        overview         = row.get("overview", "")
        overview_preview = str(overview)[:150] + "..." if overview else "No summary available"
        relevance        = row.get("relevance", 0)
        relevance_pct    = f"{relevance*100:.1f}%" if relevance else "N/A"

        print(f"{idx}. {title} ({year})")
        print(f"   Type: {type_name}")
        print(f"   Rating: {rating_str} / 10")
        print(f"   Genres: {genres}")
        if extra:
            print(f"   Details: {', '.join(extra)}")
        print(f"   Relevance: {relevance_pct}")
        print(f"   Overview: {overview_preview}")
        print("-" * 65 + "\n")


In [13]:
def get_state_summary(memory):
    type_names   = {"movie": "Film", "tv_show": "Series", "book": "Book"}
    content_type = memory.state["content_type"]

    print("\n" + "="*50)
    print("Current Search State:")
    print("="*50)
    print(f"  Mood:         {memory.state['mood'] or 'Not specified'}")
    print(f"  Topic:        {memory.state['topic'] or 'Not specified'}")
    print(f"  Content Type: {type_names.get(content_type, 'Not specified')}")
    c = memory.state["constraints"]
    if any(v for v in c.values()):
        print(f"  Constraints:")
        if c["year_from"]:    print(f"    Year from:   {c['year_from']}")
        if c["year_to"]:      print(f"    Year to:     {c['year_to']}")
        if c["min_rating"]:   print(f"    Min rating:  {c['min_rating']}")
        if c["max_runtime"]:  print(f"    Max runtime: {c['max_runtime']} min")
    print("="*50)


def show_statistics():
    print("\n" + "="*50)
    print("Dataset Statistics")
    print("="*50)
    print(f"  Total Entries: {len(df):,}")
    type_map = {"movie": "Movies", "tv_show": "TV Shows", "book": "Books"}
    for t, count in df["type"].value_counts().items():
        print(f"  {type_map.get(t, t)}: {count:,}")
    print(f"  Avg Rating: {df['rating'].mean():.2f}")
    yr = df["year"].dropna()
    print(f"  Year Range: {int(yr.min())} – {int(yr.max())}")
    print("="*50)


In [14]:
def run_chat_interface():
    memory = DialogMemory()

    print("=" * 60)
    print("🎬 Welcome to StoryVerse Finder")
    print("=" * 60)
    print("I am your assistant for finding movies, TV shows, and books.")
    print("Tell me what you are looking for in a natural way 😊")
    print("Examples:")
    print("  - بدي اشي عن شرطة وتحقيق")
    print("  - زهقانة بدي فيلم")
    print("  - بدي رواية عميقة عن غموض وجريمة")
    print("  - مسلسلات بعد 2015 تقييمها فوق 8")
    print("\nCommands: 'reset' | 'state' | 'all' | 'stats' | 'exit'")
    print("=" * 60 + "\n")

    while True:
        try:
            pre_user_input = input("User: ").strip()

            if not pre_user_input:
                continue

            # ── أوامر خاصة ──────────────────────────────────
            if pre_user_input in ["exit", "quit", "bye", "خلص"]:
                print("\nThank you for using StoryVerse Finder. Goodbye! 👋")
                break

            if pre_user_input == "reset":
                memory = DialogMemory()
                print("\nConversation reset. How are you feeling today? 😊")
                continue

            if pre_user_input == "state":
                get_state_summary(memory)
                memory.add_system_message("Displayed search state")
                continue

            if pre_user_input == "stats":
                show_statistics()
                continue

            if pre_user_input == "all":
                if memory.last_results is not None and not memory.last_results.empty:
                    print("\nDisplaying all results:")
                    display_results_formatted(memory.last_results, full=True)
                else:
                    print("\nNo previous results found. Please perform a search first.")
                continue

            # Preprocessing
            cleaned_input, moods, negated = preprocess(pre_user_input, MOOD_MAP)
            user_input = cleaned_input

            #  Update Belief State
            memory.update(user_input)

            # تحديث الـ mood من الـ negation logic
            if moods:
                mood       = moods[-1]
                boost_words = MOOD_MAP[mood]["boost"].split()
                if not any(word in negated for word in boost_words):
                    memory.state["mood"] = mood
            if not moods and negated:
                memory.state["mood"] = None

            #  Clarification
            question = memory.needs_clarification()
            if question:
                print(f"\nSystem: {question}")
                memory.add_system_message(question)
                continue

            #  Build Query (Rewriter)
            query = memory.build_query()
            print(f"\n[Search Query] {query}")

            #  Retrieve
            raw_results = search_enhanced(query, memory, top_k=200)

            #  Filter
            filtered_results = apply_filters(raw_results, memory.state)

            #  Rerank
            final_results = rerank(filtered_results, memory.state, top_k=5)

            memory.last_results = filtered_results  # للـ 'all' command

            #  Display
            if not final_results.empty:
                display_results_formatted(final_results, top_k=5)
                print("\nYou can refine your search by providing more details.")
            else:
                print("\nNo suitable results found.")
                print("Please try different search criteria or type 'reset'.")

            memory.add_system_message("Displayed search results")

        except KeyboardInterrupt:
            print("\n\nSearch interrupted. Goodbye.")
            break
        except Exception as e:
            print(f"\nError occurred: {e}")
            print("Please try again or type 'reset'")


In [15]:
run_chat_interface()

🎬 Welcome to StoryVerse Finder
I am your assistant for finding movies, TV shows, and books.
Tell me what you are looking for in a natural way 😊
Examples:
  - بدي اشي عن شرطة وتحقيق
  - زهقانة بدي فيلم
  - بدي رواية عميقة عن غموض وجريمة
  - مسلسلات بعد 2015 تقييمها فوق 8

Commands: 'reset' | 'state' | 'all' | 'stats' | 'exit'

User: زهقانة 

System: حلو! تحب فيلم، مسلسل، ولا رواية؟ 🎬📺📚
User: فيلم

[Search Query] entertaining light-hearted fun comedy film movie

Search Results (Top 5 of 5)

1. AMV Hell 3: The Motion Picture (2005)
   Type: Film
   Rating: 10.0 / 10
   Genres: Comedy, Animation, Music
   Details: 69 min
   Relevance: 6.9%
   Overview: An enormously creative and witty comedic film created by the combination of seemingly random and out of place animation clips with accompanying audio ...
-----------------------------------------------------------------

2. The SuperVips (1968)
   Type: Film
   Rating: 7.4 / 10
   Genres: Animation, Comedy
   Details: 79 min
   Relevance: 5.